In [10]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

warnings.filterwarnings("ignore")



In [11]:

# ============================================================
# 0. 설정
# ============================================================

DATA_PATH = r"..\data\Membership_train.csv"
OUTPUT_DIR = "membership_blank_baseline_outputs"

TARGET_COL = "Repurchase"
USER_ID_COL = "uno"

RANDOM_STATE = 42
N_SPLITS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [12]:


# ============================================================
# 1. 유틸
# ============================================================

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def normalize_yes_no_to_binary(series, positive_values=("Y", "O", "1", "TRUE", "True", "true")):
    s = series.copy()

    if s.dtype == "object":
        s = s.astype(str).str.strip()
        s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})

        out = s.isin(positive_values).astype(int)

        # 원래 결측은 0으로 처리
        out[series.isna()] = 0
        return out

    return s.fillna(0).astype(int)


def get_cv_splitter(y, groups=None):
    if groups is not None and pd.Series(groups).duplicated().any():
        try:
            from sklearn.model_selection import StratifiedGroupKFold

            splitter = StratifiedGroupKFold(
                n_splits=N_SPLITS,
                shuffle=True,
                random_state=RANDOM_STATE,
            )
            split_iter = splitter.split(np.zeros(len(y)), y, groups)
            split_name = "StratifiedGroupKFold"
            return split_iter, split_name

        except ImportError:
            print("[WARNING] StratifiedGroupKFold를 사용할 수 없어 StratifiedKFold로 대체합니다.")

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    split_iter = splitter.split(np.zeros(len(y)), y)
    split_name = "StratifiedKFold"
    return split_iter, split_name


def evaluate_feature_set(df, y, feature_cols, feature_set_name, groups=None):
    X = df[feature_cols].copy()

    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
    )

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
        ),
    }

    fold_rows = []
    prediction_rows = []

    for model_name, model in models.items():
        print("=" * 90)
        print(f"[FEATURE SET] {feature_set_name}")
        print(f"[MODEL] {model_name}")
        print("=" * 90)

        split_iter, split_name = get_cv_splitter(y, groups=groups)
        oof_pred = np.zeros(len(df), dtype=float)

        for fold, split_result in enumerate(split_iter, start=1):
            train_idx, valid_idx = split_result

            X_train = X.iloc[train_idx]
            X_valid = X.iloc[valid_idx]
            y_train = y.iloc[train_idx]
            y_valid = y.iloc[valid_idx]

            pipe = Pipeline(steps=[
                ("preprocess", preprocess),
                ("model", model),
            ])

            pipe.fit(X_train, y_train)

            train_pred = pipe.predict_proba(X_train)[:, 1]
            valid_pred = pipe.predict_proba(X_valid)[:, 1]

            train_auc = roc_auc_score(y_train, train_pred)
            valid_auc = roc_auc_score(y_valid, valid_pred)

            oof_pred[valid_idx] = valid_pred

            fold_rows.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "cv_splitter": split_name,
                "fold": fold,
                "train_auc": train_auc,
                "valid_auc": valid_auc,
                "gap_train_valid": train_auc - valid_auc,
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
                "valid_positive_rate": y_valid.mean(),
                "feature_count": len(feature_cols),
                "numeric_feature_count": len(numeric_cols),
                "categorical_feature_count": len(categorical_cols),
            })

            print(
                f"fold {fold} | "
                f"train_auc={train_auc:.5f} | "
                f"valid_auc={valid_auc:.5f} | "
                f"gap={train_auc - valid_auc:.5f}"
            )

        oof_auc = roc_auc_score(y, oof_pred)
        print(f"[OOF AUC] {feature_set_name} / {model_name}: {oof_auc:.5f}")
        print()

        for i, pred in enumerate(oof_pred):
            prediction_rows.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "row_index": i,
                "y_true": int(y.iloc[i]),
                "oof_pred_repurchase": float(pred),
            })

    return pd.DataFrame(fold_rows), pd.DataFrame(prediction_rows)


In [13]:


# ============================================================
# 2. 데이터 로드
# ============================================================

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print("=" * 90)
print("[DATA LOADED]")
print(f"DATA_PATH: {DATA_PATH}")
print(f"shape: {df.shape}")
print("=" * 90)
print(df.head())
print()


[DATA LOADED]
DATA_PATH: ..\data\Membership_train.csv
shape: (24074, 15)
                                                 uno productcode  pgamount  \
0  7a6960912bebe03c6e4c770eb1aa91329c3497f18f90ca...     pk_1489     100.0   
1  4ec765db76545c1d6dda9f421590bf9d02f584009f8d92...     pk_1487     100.0   
2  01b16f9f7ff29b48b1ee0d1a89d1eb9662474e5eedb8c2...     pk_1488     100.0   
3  d2f2278c38ea110d35afb55fa20ccf96d735e1254c1fd4...     pk_1487     100.0   
4  534bc7b6b1ce05de6523f79872108d8dccc1dab38e2c1f...     pk_2025    7900.0   

   chargetypeid  concurrentwatchcount promo_100 coinReceived devicetypeid  \
0           134                   4.0         O          NaN           pc   
1           190                   1.0         O            O           pc   
2           180                   2.0         O          NaN      android   
3           190                   1.0         O          NaN       mobile   
4           151                   1.0       NaN          NaN      android

In [14]:


# ============================================================
# 3. 타깃 생성
# ============================================================

if TARGET_COL not in df.columns:
    raise ValueError(f"타깃 컬럼이 없습니다: {TARGET_COL}")

y = normalize_yes_no_to_binary(df[TARGET_COL], positive_values=("Y", "1", "TRUE", "True", "true"))

target_summary = (
    y.value_counts()
    .sort_index()
    .rename_axis("is_repurchase")
    .reset_index(name="count")
)
target_summary["rate"] = target_summary["count"] / len(y)

print("[TARGET SUMMARY]")
print(target_summary)
print()


[TARGET SUMMARY]
   is_repurchase  count      rate
0              0   6749  0.280344
1              1  17325  0.719656



In [15]:


# ============================================================
# 4. 깡통 membership용 최소 파생
# ============================================================

if "promo_100" in df.columns:
    df["promo_100_bin"] = normalize_yes_no_to_binary(df["promo_100"], positive_values=("O", "Y", "1"))

if "coinReceived" in df.columns:
    df["coinReceived_bin"] = normalize_yes_no_to_binary(df["coinReceived"], positive_values=("O", "Y", "1"))

if "registerday" in df.columns:
    df["registerday_dt"] = pd.to_datetime(df["registerday"], errors="coerce")
    df["register_weekday"] = df["registerday_dt"].dt.weekday
    df["register_dayofmonth"] = df["registerday_dt"].dt.day

if "endday" in df.columns:
    df["endday_dt"] = pd.to_datetime(df["endday"], errors="coerce")

if "registerday_dt" in df.columns and "endday_dt" in df.columns:
    df["subscription_days"] = (df["endday_dt"] - df["registerday_dt"]).dt.days


In [16]:


# ============================================================
# 5. duration / 중복 사용자 기초 점검
# ============================================================

audit_rows = []

audit_rows.append({
    "check_name": "row_count",
    "value": len(df),
    "note": "전체 행 수",
})

if USER_ID_COL in df.columns:
    audit_rows.append({
        "check_name": "unique_user_count",
        "value": df[USER_ID_COL].nunique(),
        "note": "고유 uno 수",
    })
    audit_rows.append({
        "check_name": "duplicated_user_rows",
        "value": int(df[USER_ID_COL].duplicated(keep=False).sum()),
        "note": "중복 uno에 속한 행 수",
    })

if "subscription_days" in df.columns:
    audit_rows.append({
        "check_name": "duration_lt_21_count",
        "value": int((df["subscription_days"] < 21).sum()),
        "note": "21일 미만 단기 종료 후보 행 수",
    })
    audit_rows.append({
        "check_name": "duration_lt_21_rate",
        "value": float((df["subscription_days"] < 21).mean()),
        "note": "21일 미만 단기 종료 후보 비율",
    })
    audit_rows.append({
        "check_name": "duration_0_count",
        "value": int((df["subscription_days"] == 0).sum()),
        "note": "duration 0일 행 수",
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(
    os.path.join(OUTPUT_DIR, "00_membership_input_audit.csv"),
    index=False,
    encoding="utf-8-sig",
)

print("[INPUT AUDIT]")
print(audit_df.to_string(index=False))
print()


[INPUT AUDIT]
          check_name     value                note
           row_count 24074.000              전체 행 수
   unique_user_count 23679.000            고유 uno 수
duplicated_user_rows   656.000      중복 uno에 속한 행 수
duration_lt_21_count   650.000 21일 미만 단기 종료 후보 행 수
 duration_lt_21_rate     0.027  21일 미만 단기 종료 후보 비율
    duration_0_count   498.000     duration 0일 행 수



In [17]:


# ============================================================
# 6. feature set 정의
# ============================================================

base_features = [
    "productcode",
    "pgamount",
    "chargetypeid",
    "concurrentwatchcount",
    "devicetypeid",
    "isauth",
    "gender",
    "agegroup",
    "registerhour",
    "register_weekday",
    "register_dayofmonth",
]

plus_promo_features = base_features + [
    "promo_100_bin",
]

plus_promo_coin_features = plus_promo_features + [
    "coinReceived_bin",
]

with_duration_features = plus_promo_coin_features + [
    "subscription_days",
]

feature_sets = {
    "A_bare_membership_no_promo": base_features,
    "B_membership_plus_promo": plus_promo_features,
    "C_membership_plus_promo_coin": plus_promo_coin_features,
    "D_membership_plus_promo_coin_duration": with_duration_features,
}

clean_feature_sets = {}

for feature_set_name, cols in feature_sets.items():
    existing_cols = [col for col in cols if col in df.columns]

    nunique = df[existing_cols].nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    final_cols = [col for col in existing_cols if col not in constant_cols]

    clean_feature_sets[feature_set_name] = final_cols

feature_inventory_rows = []

for feature_set_name, cols in clean_feature_sets.items():
    for col in cols:
        feature_inventory_rows.append({
            "feature_set": feature_set_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "nunique": df[col].nunique(dropna=True),
            "missing_count": int(df[col].isna().sum()),
        })

feature_inventory = pd.DataFrame(feature_inventory_rows)
feature_inventory.to_csv(
    os.path.join(OUTPUT_DIR, "01_membership_baseline_feature_inventory.csv"),
    index=False,
    encoding="utf-8-sig",
)

print("[FEATURE SETS]")
for name, cols in clean_feature_sets.items():
    print(f"{name}: {len(cols)} features")
    print(cols)
    print()


[FEATURE SETS]
A_bare_membership_no_promo: 11 features
['productcode', 'pgamount', 'chargetypeid', 'concurrentwatchcount', 'devicetypeid', 'isauth', 'gender', 'agegroup', 'registerhour', 'register_weekday', 'register_dayofmonth']

B_membership_plus_promo: 12 features
['productcode', 'pgamount', 'chargetypeid', 'concurrentwatchcount', 'devicetypeid', 'isauth', 'gender', 'agegroup', 'registerhour', 'register_weekday', 'register_dayofmonth', 'promo_100_bin']

C_membership_plus_promo_coin: 13 features
['productcode', 'pgamount', 'chargetypeid', 'concurrentwatchcount', 'devicetypeid', 'isauth', 'gender', 'agegroup', 'registerhour', 'register_weekday', 'register_dayofmonth', 'promo_100_bin', 'coinReceived_bin']

D_membership_plus_promo_coin_duration: 14 features
['productcode', 'pgamount', 'chargetypeid', 'concurrentwatchcount', 'devicetypeid', 'isauth', 'gender', 'agegroup', 'registerhour', 'register_weekday', 'register_dayofmonth', 'promo_100_bin', 'coinReceived_bin', 'subscription_days']


In [18]:


# ============================================================
# 7. 모델 평가
# ============================================================

groups = df[USER_ID_COL] if USER_ID_COL in df.columns else None

all_fold_results = []
all_predictions = []

for feature_set_name, feature_cols in clean_feature_sets.items():
    fold_result, pred_result = evaluate_feature_set(
        df=df,
        y=y,
        feature_cols=feature_cols,
        feature_set_name=feature_set_name,
        groups=groups,
    )

    all_fold_results.append(fold_result)
    all_predictions.append(pred_result)

fold_results = pd.concat(all_fold_results, ignore_index=True)
predictions = pd.concat(all_predictions, ignore_index=True)



[FEATURE SET] A_bare_membership_no_promo
[MODEL] LogisticRegression
fold 1 | train_auc=0.57751 | valid_auc=0.56435 | gap=0.01316
fold 2 | train_auc=0.57793 | valid_auc=0.56322 | gap=0.01471
fold 3 | train_auc=0.57765 | valid_auc=0.56238 | gap=0.01527
fold 4 | train_auc=0.57366 | valid_auc=0.57425 | gap=-0.00059
fold 5 | train_auc=0.57599 | valid_auc=0.56781 | gap=0.00819
[OOF AUC] A_bare_membership_no_promo / LogisticRegression: 0.56601

[FEATURE SET] A_bare_membership_no_promo
[MODEL] RandomForest
fold 1 | train_auc=0.81295 | valid_auc=0.54533 | gap=0.26762
fold 2 | train_auc=0.81924 | valid_auc=0.54489 | gap=0.27435
fold 3 | train_auc=0.81950 | valid_auc=0.55193 | gap=0.26757
fold 4 | train_auc=0.81351 | valid_auc=0.56099 | gap=0.25252
fold 5 | train_auc=0.81653 | valid_auc=0.54306 | gap=0.27347
[OOF AUC] A_bare_membership_no_promo / RandomForest: 0.54887

[FEATURE SET] A_bare_membership_no_promo
[MODEL] HistGradientBoosting
fold 1 | train_auc=0.62189 | valid_auc=0.55819 | gap=0.0636

In [19]:

# ============================================================
# 8. 요약 저장
# ============================================================

summary = (
    fold_results
    .groupby(["feature_set", "model", "cv_splitter"], as_index=False)
    .agg(
        mean_train_auc=("train_auc", "mean"),
        std_train_auc=("train_auc", "std"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap_train_valid", "mean"),
        std_gap=("gap_train_valid", "std"),
        feature_count=("feature_count", "first"),
        numeric_feature_count=("numeric_feature_count", "first"),
        categorical_feature_count=("categorical_feature_count", "first"),
    )
    .sort_values(["mean_valid_auc", "mean_gap"], ascending=[False, True])
)

fold_results.to_csv(
    os.path.join(OUTPUT_DIR, "02_membership_baseline_fold_results.csv"),
    index=False,
    encoding="utf-8-sig",
)

summary.to_csv(
    os.path.join(OUTPUT_DIR, "03_membership_baseline_summary.csv"),
    index=False,
    encoding="utf-8-sig",
)

predictions.to_csv(
    os.path.join(OUTPUT_DIR, "04_membership_baseline_oof_predictions.csv"),
    index=False,
    encoding="utf-8-sig",
)

target_summary.to_csv(
    os.path.join(OUTPUT_DIR, "05_membership_baseline_target_summary.csv"),
    index=False,
    encoding="utf-8-sig",
)

print("=" * 90)
print("[FINAL SUMMARY]")
print(summary.to_string(index=False))
print("=" * 90)
print()

print("[SAVED FILES]")
for filename in [
    "00_membership_input_audit.csv",
    "01_membership_baseline_feature_inventory.csv",
    "02_membership_baseline_fold_results.csv",
    "03_membership_baseline_summary.csv",
    "04_membership_baseline_oof_predictions.csv",
    "05_membership_baseline_target_summary.csv",
]:
    print(os.path.join(OUTPUT_DIR, filename))

[FINAL SUMMARY]
                          feature_set                model          cv_splitter  mean_train_auc  std_train_auc  mean_valid_auc  std_valid_auc  mean_gap  std_gap  feature_count  numeric_feature_count  categorical_feature_count
D_membership_plus_promo_coin_duration   LogisticRegression StratifiedGroupKFold        0.583700       0.002296        0.573998       0.006254  0.009702 0.008529             14                     10                          4
         C_membership_plus_promo_coin   LogisticRegression StratifiedGroupKFold        0.582871       0.002103        0.572950       0.005693  0.009921 0.007782             13                      9                          4
D_membership_plus_promo_coin_duration HistGradientBoosting StratifiedGroupKFold        0.653829       0.028224        0.569990       0.005414  0.083840 0.028685             14                     10                          4
         C_membership_plus_promo_coin HistGradientBoosting StratifiedGroupKFold 